In [1]:
%load_ext autoreload
%autoreload 2

from reader import read_csv
from transformer import snake_case_columns, map_columns, COLUMN_MAPPING, convert_types
from validator import find_missing_columns

typed = convert_types(map_columns(snake_case_columns(read_csv("olist_data/order_items.csv")), COLUMN_MAPPING))

print(find_missing_columns(typed, ["order_id", "item_price"]))
print(find_missing_columns(typed, ["order_id", "not_a_column"]))

!pytest -v

[]
['not_a_column']
============================= test session starts =============================
platform win32 -- Python 3.14.10, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Kishore\Documents\Project\tap-python\venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Kishore\Documents\Project\tap-python
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 3 items

tests/test_transformer.py::test_to_snake_case_converts_spaces_camel_case_and_punctuation PASSED [ 33%]
tests/test_transformer.py::test_to_float_converts_strings_and_treats_blank_as_none PASSED [ 66%]
tests/test_transformer.py::test_calculate_order_totals_sums_price_and_freight_per_order PASSED [100%]

============================== 3 passed in 0.02s ==============================


Task 1: write read_csv here, then move it to reader.py

In [2]:
import csv

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

rows = read_csv("olist_data/order_items.csv")
print(len(rows), "rows")

200 rows


In [3]:
import inspect
from transformer import to_snake_case, convert_types

print(inspect.getsource(to_snake_case))
print(inspect.getsource(convert_types))

def to_snake_case(name: str) -> str:
    """Convert a column name to snake_case.

    Examples:
        "Order ID" -> "order_id"
        "freightValue" -> "freight_value"
        "order-item.id" -> "order_item_id"

    Args:
        name: The original column name.

    Returns:
        The name in lowercase snake_case.
    """
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name.strip())
    s = re.sub(r"[\s\-.]+", "_", s)
    s = re.sub(r"[^0-9A-Za-z_]", "", s)
    s = re.sub(r"_+", "_", s)
    return s.lower().strip("_")

def convert_types(rows: list[dict[str, str]]) -> list[dict]:
    """Convert price and freight to float and timestamps to datetime.

    Columns that aren't present in a row are skipped.

    Args:
        rows: Rows with standard column names, as returned by map_columns.

    Returns:
        A new list of dictionaries with converted values.
    """
    converted = []
    for row in rows:
        new_row = dict(row)
        for col in FLOAT_COLUMNS:
            if co